# Regularization Experiments
**Hypothesis:** Dropout prevents overfitting | **Result:** Dropout hurts minority class

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Data and utilities
from utils.data_loadingV2 import load_data
from utils.band_mapping import load_band_mapping
from utils.training import train_two_stage
from utils.reproducibility_utils import seed_everything
from utils.visualization import plot_sweep_results

# Models from models/ folder
from models.vanilla_rnn import VanillaRNN
from models.simple_lstm import SimpleLSTM

SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")

Device: cuda
Project root: /home/mdesouky/workspace/mait_g2_drift_challenge


In [2]:
# Load data
TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
COLUMNS_TO_DROP = ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time']

train_loader, val_loader, scaler, _, _ = load_data(
    train_path=str(TRAIN_PATH), val_ratio=0.15, columns_to_drop=COLUMNS_TO_DROP,
    sequence_length=20, target="sideSlip", threshold=True, threshold_value=3.0,
    threshold_column="vxCG", batchsize=64, scaling=True, seed=SEED, shuffle=True
)
band_dict = load_band_mapping()
print(f"Train: {len(train_loader)} batches")

V5 Data Loader: Stratified Temporal Split

Processing 22 files...
Total groups after threshold: 42

Train: Many=90.3%, Medium=7.6%, Few=2.1%
Val:   Many=96.6%, Medium=2.3%, Few=1.1%

Train: 120,445 sequences | Val: 16,635 sequences

Train: 1882 batches


## 1. RNN Dropout Sweep

In [3]:
dropout_results = []
for dropout in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    seed_everything(SEED)
    print(f"Training RNN dropout={dropout}...")
    model = VanillaRNN(22, 64, 1, dropout)
    _, metrics = train_two_stage(model, train_loader, val_loader, band_dict, device)
    print(f"  Overall={metrics['overall_mae']:.4f}, Few={metrics['few_mae']:.4f}")
    dropout_results.append({'dropout': dropout, **metrics})

pd.DataFrame(dropout_results)

Training RNN dropout=0.0...


  Overall=0.1436, Few=0.2217
Training RNN dropout=0.1...
  Overall=0.1357, Few=0.5666
Training RNN dropout=0.2...
  Overall=0.1524, Few=1.2220
Training RNN dropout=0.3...
  Overall=0.1932, Few=1.7057
Training RNN dropout=0.4...
  Overall=0.2504, Few=2.3000
Training RNN dropout=0.5...
  Overall=0.3272, Few=3.0065


,dropout,many_mae,many_count,many_pct,medium_mae,medium_count,medium_pct,few_mae,few_count,few_pct,overall_mae
0,0.0,0.129681,4230,25.428314,0.139673,95,0.571085,0.221694,18,0.108206,0.143571
1,0.1,0.114068,4230,25.428314,0.321493,95,0.571085,0.566642,18,0.108206,0.135658
2,0.2,0.112052,4230,25.428314,0.658722,95,0.571085,1.222014,18,0.108206,0.152406
3,0.3,0.132359,4230,25.428314,1.001163,95,0.571085,1.705656,18,0.108206,0.193208
4,0.4,0.154901,4230,25.428314,1.424230,95,0.571085,2.300027,18,0.108206,0.250374
5,0.5,0.198591,4230,25.428314,1.917331,95,0.571085,3.006543,18,0.108206,0.327218


## 2. LSTM Dropout Comparison

In [4]:
lstm_results = []
for dropout in [0.0, 0.2, 0.4]:
    seed_everything(SEED)
    print(f"Training LSTM dropout={dropout}...")
    model = SimpleLSTM(22, 64, 1, dropout)
    _, metrics = train_two_stage(model, train_loader, val_loader, band_dict, device)
    print(f"  Overall={metrics['overall_mae']:.4f}")
    lstm_results.append({'dropout': dropout, **metrics})

pd.DataFrame(lstm_results)

Training LSTM dropout=0.0...
  Overall=0.1416
Training LSTM dropout=0.2...
  Overall=0.1292
Training LSTM dropout=0.4...
  Overall=0.1245


,dropout,many_mae,many_count,many_pct,medium_mae,medium_count,medium_pct,few_mae,few_count,few_pct,overall_mae
0,0.0,0.129137,4230,25.428314,0.148118,95,0.571085,0.185895,18,0.108206,0.141608
1,0.2,0.115997,4230,25.428314,0.178824,95,0.571085,0.180551,18,0.108206,0.129202
2,0.4,0.111031,4230,25.428314,0.168475,95,0.571085,0.203892,18,0.108206,0.124516


## Visualization

In [5]:
# Using visualization from utils/visualization.py
df = pd.DataFrame(dropout_results)

# Overall MAE vs Dropout
fig_overall = plot_sweep_results(df, x_col='dropout', y_col='overall_mae', 
                                  title='Dropout Impact: Overall MAE', highlight_optimal=True)
fig_overall.show()

# Few-shot MAE vs Dropout
fig_few = plot_sweep_results(df, x_col='dropout', y_col='few_mae',
                              title='Dropout Impact: Few-shot MAE', highlight_optimal=True)
fig_few.show()

## Summary

| Issue | Explanation |
|-------|-------------|
| Signal attenuation | Few-shot (5%) × (1-p)³ ≈ 1.7% effective signal |
| Data scarcity | Stochastic masking worsens minority class learning |

**Conclusion:** Constrain model capacity instead of using dropout for imbalanced regression.

## Interactive Predictions vs Ground Truth

In [6]:
# Interactive line plot showing predictions overlaid on ground truth
from utils.visualization import plot_predictions_vs_truth
from utils.training import get_predictions

# Get predictions from best model (dropout=0.0)
seed_everything(SEED)
best_model = VanillaRNN(22, 64, 1, dropout=0.0)
best_model, _ = train_two_stage(best_model, train_loader, val_loader, band_dict, device)
val_preds, val_targets = get_predictions(best_model, val_loader, device)

fig_line = plot_predictions_vs_truth(
    val_preds, val_targets,
    title='RNN (No Dropout): Predictions vs Ground Truth',
    max_samples=5000
)
fig_line.show()